# Autoregressive Generation and Decoding

> Training has taught the model to score every token in its vocabulary. During generation, however, each forward pass produces only logits. One final step must select the next token from tens of thousands of candidates.
>
> This chapter answers one question: **how do scores become a token?** We cover softmax probabilities; Greedy, Temperature, Top-k, and Top-p; Repetition Penalty and Beam Search; and the order in which these rules form one sampling step.

After reading, parameters such as `temperature`, `top_p`, `top_k`, and `repetition_penalty` should describe concrete changes to the selection rule rather than changes to the model itself.

Suppose the vocabulary has six candidates after the prompt “The capital of France is”:

```text
Paris    4.2
London   3.8
Beijing  1.2
Tokyo    0.8
period   0.4
banana  -0.5
```

Choosing Paris seems sufficient, but generation may need diversity, repetition control, or consideration of several paths. Each need becomes a selection rule. We begin by interpreting logits.


## 1. From Logits to Probabilities

Logits are unnormalized scores. They need not sum to 1 and may be negative, so they support comparisons but are not probabilities.

Softmax exponentiates each score and divides by the sum. Exponentiation makes all values positive and turns score differences into probability ratios. Paris and London differ by 0.4 logits, so Paris receives $e^{0.4} \approx 1.5$ times London's probability.


In [ ]:
import torch
import torch.nn.functional as F

tokens = ["Paris", "London", "Beijing", "Tokyo", "banana", "."]
logits = torch.tensor([4.2, 3.8, 1.2, 0.8, -0.5, 0.4])

probs = F.softmax(logits, dim=-1)
for t, l, p in zip(tokens, logits, probs):
    print(f"{t:>7}  logit={l:>4.1f}  p={p.item():.3f}")

print()
print(f"Key observation: the logits for Paris and London differ by only 0.4, yet their probability ratio is "
      f"{float(probs[0] / probs[1]):.2f} (= e^0.4)")
print(f"A banana logit of -0.5 may not look tiny, but its probability is only {float(probs[4]):.4f}; exponentiation magnifies the gap.")


Two details matter. First, a logit difference becomes a multiplicative probability ratio: the difference between `banana=-0.5` and `period=0.4` is less than one logit but about a 2.5-fold probability ratio. Second, probabilities now sum to 1. Selecting a token has become sampling from a probability distribution; the following strategies define how to sample.


## 2. Greedy Decoding

The simplest rule chooses the highest-probability token with `argmax` at every step. It is deterministic, reproducible, easy to debug, and adds no sampling work. Tasks with a narrow correct answer, such as factual completion or constrained code, often use it.

Determinism can also be monotonous. For a creative prompt, Greedy always returns the same continuation even when several lower-ranked candidates are reasonable. Temperature changes the shape of the distribution before selection.


In [ ]:
next_id = torch.argmax(logits).item()
print("Greedy chooses:", tokens[next_id])

print()
print("Key observation: argmax is deterministic—the same logits choose the same Token even after ten thousand repetitions.")
print("London, the runner-up, never gets a chance even though it trails by only 0.4.")


## 3. Temperature and Distribution Shape

Temperature divides all logits by $T$ before softmax. A value below 1 magnifies score differences and makes the distribution sharper, approaching Greedy behavior. A value above 1 flattens differences, gives lower-ranked candidates more probability, and increases diversity.

**Temperature never removes a candidate.** Even at high temperature, an implausible token such as `banana` retains nonzero probability. This motivates truncation.


In [ ]:
for T in [0.2, 0.7, 1.0, 1.5]:
    p = F.softmax(logits / T, dim=-1)
    print(f"T={T:<3}: " + ", ".join(f"{t}:{x:.2f}" for t, x in zip(tokens, p.tolist())))

print()
print("Key observation: at T=0.2 the top two nearly monopolize the distribution; at T=1.5 the candidates are flatter.")
print("No matter how high T is, banana still has nonzero probability—a high temperature exposes the long tail.")


## 4. Top-k and Top-p Truncation

High temperature can give implausible tail tokens meaningful probability. Truncation keeps reasonable alternatives while removing the tail.

**Top-k** retains the $k$ highest-scoring candidates and sets all other logits to $-\infty$. With `k=2`, sampling considers only Paris and London.

**Top-p**, or Nucleus Sampling, adapts to the distribution's shape. Sort candidates by probability and keep the smallest prefix whose cumulative mass reaches $p$:

```text
A 0.50   cumulative 0.50
B 0.25   cumulative 0.75
C 0.15   cumulative 0.90  <- reaches 0.90
D 0.06   cumulative 0.96
```

For `top_p=0.9`, A, B, and C remain. A sharp distribution automatically keeps fewer candidates; a flat distribution keeps more.


In [ ]:
def top_k_filter(logits, k):
    if k is None or k >= logits.numel():
        return logits
    threshold = torch.topk(logits, k).values[-1]
    return torch.where(logits < threshold, torch.tensor(float("-inf")), logits)

def top_p_filter(logits, p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    remove = cumulative > p
    remove[1:] = remove[:-1].clone()
    remove[0] = False
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out[sorted_idx] = sorted_logits
    return out

cases = [
    ("top_k=2", top_k_filter(logits.clone(), 2)),
    ("top_p=0.9", top_p_filter(logits.clone(), 0.9)),
]
for name, filtered in cases:
    p = F.softmax(filtered, dim=-1)
    kept = [(t, round(x, 3)) for t, x in zip(tokens, p.tolist()) if x > 0]
    print(name, kept)

print()
print("Key observation: top_k=2 retains only Paris and London; top_p=0.9 retains tokens until cumulative probability covers 0.9.")


## 5. Repetition Penalty

The preceding strategies consider only the current distribution, not the generated history. A high-probability phrase may therefore repeat indefinitely.

Repetition Penalty lowers logits for tokens that have already appeared. Hugging Face divides a positive repeated-token logit by $c$ and multiplies a negative one by $c$, where $c>1$.

If Paris has already appeared and $c=1.3$, its logit falls from 4.2 to about 3.23, while London's remains 3.8. Their ranking flips. The penalty does not delete repeated tokens; it temporarily makes them less competitive.


In [ ]:
def apply_repetition_penalty(logits, generated_ids, penalty):
    """Lower logits of seen tokens: divide positive values by penalty and multiply negative ones."""
    z = logits.clone()
    for i in set(generated_ids):
        if z[i] > 0:
            z[i] = z[i] / penalty
        else:
            z[i] = z[i] * penalty
    return z

generated_ids = [tokens.index("Paris")]  # Paris has appeared once
z_new = apply_repetition_penalty(logits, generated_ids, penalty=1.3)

for i, (t, old, new) in enumerate(zip(tokens, logits, z_new)):
    mark = "  <- seen, lowered" if i in generated_ids else ""
    print(f"{t:>7}  original {old:>5.2f}  penalized {new:>5.2f}{mark}")

print()
print("Key observation: argmax changes from ", tokens[logits.argmax()], " to ", tokens[z_new.argmax()], "")


In [ ]:
# The plot makes it clear: only the Token that has appeared is lowered
import matplotlib.pyplot as plt

labels = ["Paris", "London", "Beijing", "Tokyo", "banana", "."]
xs = range(len(labels))
width = 0.38

plt.figure(figsize=(7, 3.5))
plt.bar([i - width / 2 for i in xs], logits.tolist(), width=width,
        label="original logits")
plt.bar([i + width / 2 for i in xs], z_new.tolist(), width=width,
        label="after repetition penalty")
plt.xticks(list(xs), labels)
plt.ylabel("logit")
plt.title("Only the repeated token (Paris) gets pushed down")
plt.legend()
plt.show()


## 6. Beam Search

Greedy picks the best at each step, but local optimum does not equal global optimum. Beam Search maintains K paths simultaneously and selects the K highest-scoring paths from all candidates at each step.

Suitable for: translation, summarization, and other tasks with "clear answers". Not suitable for: creative writing -- beam search makes the output boring.

In [ ]:
# Verify with the same numbers: the first choice at step one need not win over the whole path
step1 = {"A": 0.6, "B": 0.4}
step2 = {"A": {"x": 0.5, "y": 0.5}, "B": {"x": 0.9, "y": 0.1}}

greedy_first = max(step1, key=step1.get)
greedy_total = step1[greedy_first] * max(step2[greedy_first].values())
print(f"Greedy chooses {greedy_first} first, then follows the local optimum, for a total score of {greedy_total:.2f}")

all_paths = {(t1, t2): p1 * p2
             for t1, p1 in step1.items()
             for t2, p2 in step2[t1].items()}
best_path = max(all_paths, key=all_paths.get)
print("All complete paths:", all_paths)
print("Key observation: the best complete path is", "".join(best_path),
      "with score", all_paths[best_path])


In [ ]:
# Draw the two-step search tree: thicker lines mean higher probability; green is the full path found by Beam
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
nodes = {"start": (0, 0.5), "A": (1, 0.78), "B": (1, 0.22),
         "Ax": (2, 0.95), "Ay": (2, 0.62), "Bx": (2, 0.38), "By": (2, 0.05)}
edges = [("start", "A", 0.6), ("start", "B", 0.4),
         ("A", "Ax", 0.5), ("A", "Ay", 0.5),
         ("B", "Bx", 0.9), ("B", "By", 0.1)]

for a, b, p in edges:
    (x1, y1), (x2, y2) = nodes[a], nodes[b]
    on_best = (a, b) in {("start", "B"), ("B", "Bx")}
    ax.plot([x1, x2], [y1, y2], color="tab:green" if on_best else "tab:gray",
            linewidth=1 + 8 * p, alpha=1.0 if on_best else 0.45, zorder=1)
    ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.04, f"{p:.1f}",
            ha="center", fontsize=9)

for name, (x, y) in nodes.items():
    ax.text(x, y, name, ha="center", va="center", zorder=2,
            bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black"))

ax.text(*nodes["Ax"], "  A-x total 0.30\n  (greedy's choice)", va="center", fontsize=9)
ax.text(*nodes["Bx"], "  B-x total 0.36\n  (best, beam finds it)", va="center", fontsize=9)
ax.set_xlim(-0.2, 3.6)
ax.set_ylim(-0.1, 1.15)
ax.axis("off")
ax.set_title("Beam width 2: the best full path may start with the runner-up")
plt.show()


## 7. A Complete Sampling Step

A typical Decode step applies the rules in this order:

```text
logits
  -> repetition / presence / frequency penalties (use history)
  -> temperature (reshape distribution)
  -> top-k / top-p / min-p (truncate candidates)
  -> multinomial sampling
```

Penalties must inspect generated history before selection. Temperature and truncation both alter the distribution before sampling. Exact processor ordering differs across frameworks, so production behavior should be checked against the implementation in use.


In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None, seed=0):
    torch.manual_seed(seed)
    x = logits.clone() / max(temperature, 1e-5)
    x = top_k_filter(x, top_k)
    if top_p is not None:
        x = top_p_filter(x, top_p)
    probs = F.softmax(x, dim=-1)
    return torch.multinomial(probs, 1).item(), probs

print("greedy:", tokens[torch.argmax(logits).item()])
for name, cfg in [
    ("T=0.7, p=0.9", dict(temperature=0.7, top_p=0.9)),
    ("T=1.2, p=0.95", dict(temperature=1.2, top_p=0.95)),
]:
    picks = [tokens[sample_next(logits, seed=s, **cfg)[0]] for s in range(8)]
    print(name, picks)


## 8. Common Generation Parameters

| Parameter | Effect |
|:---|:---|
| `temperature` | Makes the probability distribution sharper or flatter |
| `top_k` | Keeps only the $k$ highest-scoring candidates |
| `top_p` | Dynamically truncates by cumulative probability mass |
| `repetition_penalty` | Lowers scores of tokens that already appeared |
| `max_tokens` / `max_new_tokens` | Limits generated length |
| `stop` / EOS | Defines stopping conditions |
| `seed` | Seeds sampling randomness |

When reading an API or model card, ask whether each parameter changes scores, reshapes probabilities, truncates candidates, or controls stopping.


## Summary

- Difference between inference and training: training has answers (parallel), inference has no answers (serial)
- Greedy picks the highest probability -- deterministic but uninteresting
- Temperature controls randomness -- low temperature is deterministic, high temperature is diverse
- Pure Sampling samples directly from the full vocabulary; simple but prone to drawing irrelevant tokens
- Top-k fixes the candidate count at k; k does not adapt to the distribution, suboptimal when it is very sharp or very flat
- Top-p truncates adaptively by cumulative probability -- fewer candidates when concentrated, more when dispersed; the industrial mainstream
- Top-k + Top-p together is the industrial default: fix an upper bound first, then adapt
- Beam Search maintains multiple paths -- suitable for translation/summarization
- Repetition Penalty breaks repetitive loops
- Chat templates stitch multi-turn conversations into model-readable text
- System Prompt guides the model through role and format requirements
- Common sampling execution order: penalty -> temperature -> top-k/p -> sample -> eos; the exact order depends on the framework implementation

Next section: break down the causes of slow inference, learn acceleration techniques like KV Cache, quantization, and FlashAttention.

## Exercises

**Exercise 1: Temperature Calculation**

Given logits = [2.0, 1.0, 0.5] and temperature = 0.5, what are the scaled logits?

Hint: logits / temperature

### Exercise 1: Implement Repetition Penalty

Change only tokens that already appeared: divide positive logits by the penalty and multiply negative logits by it.

Hint: iterate over indices in `generated_ids` and branch on the sign of `z[i]`.


In [ ]:
# Exercise 1: fill in repetition penalty

test_logits = torch.tensor([4.2, 3.8, -0.5, 0.4])
test_generated = [0]  # token at index 0 has already appeared

def penalize(logits, generated_ids, penalty):
    """Return new logits after lowering tokens that have appeared."""
    z = logits.clone()
    for i in set(generated_ids):
        # TODO: replace the triple-quoted text with your code
        """Divide or multiply z[i] according to its sign."""
    return z

result = penalize(test_logits, test_generated, 1.2)
assert torch.isclose(result[0], torch.tensor(4.2 / 1.2)), result
assert torch.isclose(result[1], torch.tensor(3.8)), result
print("✅ Exercise 1 passed: you implemented the basic repetition-penalty rule.")


### Exercise 2: Count Candidates Retained by Top-p

Given probabilities sorted from high to low, retain the smallest prefix whose cumulative probability reaches $p$.

Hint: accumulate probabilities until the first position where the sum is at least $p$; keep through that position.


In [ ]:
# Exercise 2: fill in the top-p candidate count

probs = torch.tensor([0.50, 0.25, 0.15, 0.06, 0.04])  # already sorted from high to low

def top_p_keep_count(probs, p):
    """Return the number of candidates retained after top-p truncation."""
    # TODO: replace the triple-quoted text with your code
    """Use cumulative probability to calculate how many candidates to keep."""

assert top_p_keep_count(probs, 0.5) == 1
assert top_p_keep_count(probs, 0.9) == 3
assert top_p_keep_count(probs, 0.99) == 5
print("✅ Exercise 2 passed: you understand that top-p dynamically truncates by probability mass.")


### Exercise 3: Measure Temperature with Entropy

A flatter distribution is harder to predict. Entropy quantifies this uncertainty.

Hint: compute `F.softmax(logits / T, dim=-1)`, then use $-\sum_i p_i \log p_i$.


In [ ]:
# Exercise 3: fill in entropy at different Temperatures

def entropy(probs):
    """Calculate distribution entropy: higher means flatter and less predictable sampling."""
    p = probs[probs > 0]
    return float(-(p * p.log()).sum())

def probs_at_temperature(T):
    # TODO: replace the triple-quoted text with your code
    """Return the softmax distribution after dividing logits by T."""

assert entropy(probs_at_temperature(0.3)) < entropy(probs_at_temperature(1.0))
assert entropy(probs_at_temperature(1.0)) < entropy(probs_at_temperature(3.0))
print("✅ Exercise 3 passed: you quantified why low temperature is sharper and high temperature is flatter.")
